# 02 — Expensive causal ground truth

For every verified held-out pair, this notebook performs signed, unclipped, two-direction full-residual interchange at the frozen L16 concept token. The matched engine response scale is reused for its idle dashboard control. This artifact is the expensive truth; cheap READ is computed separately in notebook 03.

**Output:** `artifacts/final/02_causal.json`.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
ARTIFACTS = ROOT / 'artifacts' / 'final'

from src.causal_read import compute_base_causal_rows, summarize_base_causal_sanity
from src.jlens_interface import load_model, release_model

def load_json(path: Path) -> dict:
    return json.loads(path.read_text())

def write_json(path: Path, value: object) -> None:
    path.write_text(json.dumps(value, indent=2, sort_keys=True) + '\n')

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

dataset_path = ARTIFACTS / '01_dataset.json'
clean_path = ARTIFACTS / '01_clean_manifest.json'
dataset = load_json(dataset_path)
clean = load_json(clean_path)
verified = [row for row in clean['rows'] if row['verification_status'] == 'VERIFIED']
if len(verified) != 77:
    raise RuntimeError(f'Expected 77 verified pairs, got {len(verified)}')
layer = int(clean['selection']['layer'])

bundle = load_model()
try:
    rows = compute_base_causal_rows(bundle, verified, layer=layer)
finally:
    del bundle
    release_model()

sanity = summarize_base_causal_sanity(rows)
artifact = {
    'schema_version': 'symmetric-causal-ground-truth-clean-v1',
    'protocol_sha256': clean['protocol_sha256'],
    'model': clean['model'],
    'selected_layer': layer,
    'position_rule': clean['selection']['position_rule'],
    'source_dataset': {
        'path': str(dataset_path.relative_to(ROOT)),
        'sha256': sha256(dataset_path),
    },
    'causal_sanity': sanity,
    'rows': rows,
}
output_path = ARTIFACTS / '02_causal.json'
write_json(output_path, artifact)
print(json.dumps({'causal_sanity': sanity, 'sha256': sha256(output_path)}, indent=2))


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

{
  "causal_sanity": {
    "n_pairs": 77,
    "engine_C_median": 0.9127144298688193,
    "engine_abs_C_median": 0.9127144298688193,
    "dashboard_C_median": -0.0020429009193054137,
    "dashboard_abs_C_median": 0.005082592121982211,
    "engine_sharp_directional_disagreement": 0,
    "dashboard_sharp_directional_disagreement": 0
  },
  "sha256": "004758d0a7e5f856d0210fbaa3d221a85060ca0aecfaf0236dfdbd4ca3beae21"
}
